In [1]:
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent

if not (root / "src").is_dir():
    raise FileNotFoundError("Could not find the repository folder containing src/")

sys.path.insert(0, str(root))

In [2]:
import numpy as np
import pandas as pd
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score
from src.data_load_ionosphere import load_ionosphere
from src.preprocess import preprocess

In [3]:
qubits = 4
seeds = [0, 1, 2, 3, 4]          # try [0] first to check timing
C_grid = {"C": [0.1, 1, 10, 100, 1000]}
rbf_grid = {"C": [0.1, 1, 10, 100, 1000], "gamma": [0.001, 0.01, 0.1, 1, 10, "scale"]}

In [4]:
ranges = [1.0, 0.5]              # angle-range upper bounds to test
feature_map = zz_feature_map(feature_dimension=qubits, reps=2, entanglement="linear")
kernel = FidelityQuantumKernel(feature_map=feature_map)

In [5]:
def metrics(y_true, pred):
    return (accuracy_score(y_true, pred),
            recall_score(y_true, pred, pos_label=0),
            f1_score(y_true, pred, average="macro"))

In [6]:
rows = []
for seed in seeds:
    X_train, X_test, y_train, y_test = load_ionosphere(seed)
    _, _, X_train_pca, X_test_pca, pca = preprocess(X_train, X_test, qubits)

    for upper in ranges:

        mm = MinMaxScaler(feature_range=(0, upper))
        X_train_quantum = mm.fit_transform(X_train_pca)
        X_test_quantum = np.clip(mm.transform(X_test_pca), 0, upper)

        K_train = kernel.evaluate(X_train_quantum, X_train_quantum)
        K_test = kernel.evaluate(X_test_quantum, X_train_quantum)

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        q = GridSearchCV(SVC(kernel="precomputed"), C_grid, cv=cv, scoring="f1_macro").fit(K_train, y_train)
        acc, rec, f1 = metrics(y_test, q.predict(K_test))
        rows.append({"seed": seed, "model": f"QSVM ZZ (0,{upper})", "acc": acc, "recall_bad": rec, "f1": f1})

        r = GridSearchCV(SVC(kernel="rbf"), rbf_grid, cv=cv, scoring="f1_macro").fit(X_train_quantum, y_train)
        acc, rec, f1 = metrics(y_test, r.predict(X_test_quantum))
        rows.append({"seed": seed, "model": f"RBF tuned (same input as ZZ (0,{upper}))", "acc": acc, "recall_bad": rec, "f1": f1})

    print(f"seed {seed} done", flush=True)

    pd.DataFrame(rows).to_csv("ionosphere_zz.csv", index=False)

seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done


In [7]:
df = pd.DataFrame(rows)
print(df.groupby("model")[["acc", "recall_bad", "f1"]].agg(["mean", "std"]).round(3).to_string())

                                        acc        recall_bad            f1       
                                       mean    std       mean    std   mean    std
model                                                                             
QSVM ZZ (0,0.5)                       0.868  0.043      0.736  0.108  0.848  0.053
QSVM ZZ (0,1.0)                       0.859  0.059      0.736  0.128  0.839  0.070
RBF tuned (same input as ZZ (0,0.5))  0.870  0.056      0.728  0.131  0.850  0.070
RBF tuned (same input as ZZ (0,1.0))  0.870  0.050      0.736  0.108  0.851  0.061
